# GLI3: Pallister-Hall vs Greig cephalopolysyndactyly

Cohort: `GLI3`, Phenopacket Store v0.1.27 (n = 81).

Disease composition:
- Greig cephalopolysyndactyly syndrome (GCPS), [OMIM:175700](https://omim.org/entry/175700), n = 50
- Pallister-Hall syndrome (PHS), [OMIM:146510](https://omim.org/entry/146510), n = 21
- Postaxial polydactyly, types A1 and B, [OMIM:174200](https://omim.org/entry/174200), n = 10

GCPS and PHS are distinct entities caused by variants in the same gene, and the split is
predictable from variant type and position: haploinsufficiency causes GCPS, whereas
truncating variants in the middle third of the gene (3' of the zinc-finger domain) cause
PHS (Johnston et al. 2005, PMID 15739154; Demurger et al. 2015, Eur J Hum Genet).
Disease membership is therefore a well-grounded synergy target.


In [ ]:
from ppktstore.registry import configure_phenopacket_registry
from phenosign import (
    PhenotypeDatasetBuilder,
    HPOCorrelationAnalyzer,
    SynergyAnalyzer,
    has_disease,
)
import pandas as pd

In [ ]:
registry = configure_phenopacket_registry()
with registry.open_phenopacket_store(release="0.1.27") as ps:
    phenopackets = list(ps.iter_cohort_phenopackets("GLI3"))

print(f"Loaded {len(phenopackets)} phenopackets")

In [ ]:
dataset = PhenotypeDatasetBuilder(phenopackets).build(build_gpsea_cohort=False)

In [ ]:
diseases_df, sex_df, genes_df, _ = dataset.describe_conditions()
diseases_df

## Pairwise correlation

In [ ]:
analyzer = HPOCorrelationAnalyzer(dataset=dataset)
results = analyzer.compute_correlation_matrix()

results.results_table.head()

In [ ]:
results.save_correlation_results(
    output_file="../case_studies/GLI3/correlation_results_GLI3.csv"
)

In [ ]:
sig = results.results_table.query("adj_p_value < 0.05")
sig = sig.reindex(sig["correlation"].abs().sort_values(ascending=False).index)
sig[["HPO_A_label", "HPO_B_label", "correlation", "adj_p_value", "n_individuals", "n_pmids"]]

In [ ]:
results.plot_correlation_heatmap_with_significance(
    title_name="Cohort GLI3",
)

## Synergy with respect to disease subtype

Target: Pallister-Hall syndrome (OMIM:146510) as the positive class; GCPS and postaxial
polydactyly form the negative class.

In [ ]:
condition = dataset.get_condition(
    has_disease("OMIM:146510"),
    name="Pallister-Hall vs other GLI3",
)

condition.value_counts(dropna=False)

In [ ]:
synergy_analyzer = SynergyAnalyzer(
    dataset=dataset,
    min_individuals_for_synergy_calculation=20,
)
synergy_results = synergy_analyzer.compute_synergy_matrix(condition=condition)

synergy_results.results_table.head()

In [ ]:
# save_synergy_results() filters at synergy >= 0.01 and drops the negative
# (redundant) half of the distribution. Save the full table as well.
synergy_results.results_table.to_csv(
    "../case_studies/GLI3/synergy_full_GLI3_OMIM_146510.csv", index=False
)
synergy_results.save_synergy_results(
    output_file="../case_studies/GLI3/synergy_matrix_GLI3_OMIM_146510.csv"
)

In [ ]:
t = synergy_results.results_table
print("Top synergy")
display(t.nlargest(10, "synergy")[["HPO_A_label", "HPO_B_label", "synergy", "adj_p_value", "n_individuals"]])

print("Top dissynergy (redundancy)")
display(t.nsmallest(10, "synergy")[["HPO_A_label", "HPO_B_label", "synergy", "adj_p_value", "n_individuals"]])

In [ ]:
synergy_results.plot_synergy_heatmap()

## Optional: reverse contrast

GCPS as the positive class gives a better balanced split (50 vs 31) and can be used to
check which pairs are synergistic in the opposite direction.

In [ ]:
condition_gcps = dataset.get_condition(
    has_disease("OMIM:175700"),
    name="GCPS vs other GLI3",
)
synergy_gcps = synergy_analyzer.compute_synergy_matrix(condition=condition_gcps)
synergy_gcps.results_table.nlargest(10, "synergy")[
    ["HPO_A_label", "HPO_B_label", "synergy", "adj_p_value", "n_individuals"]
]